## Result filter module - _Attention-Retrieval (AR)_ - Evaluation

## Initialization

In [1]:
import sys
import os
if 'google.colab' in sys.modules:
	# !pip install --upgrade datasets sentence_transformers
	!pip install sentence_transformers
	from IPython.display import clear_output
	clear_output()
else:
	# if not in 'notebooks' directory, change to it
	if not os.getcwd().endswith('Result-filter-RL'):
		os.chdir('notebooks')
		os.chdir('Result-filter-RL')

## Load the model and dataset

In [2]:
from sentence_transformers import CrossEncoder

# Model selected from the initial evaluation
model_name = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # 22.7M params
# 3.3M downloads on Huggingface last month
model_name_short = 'MiniLM-L6-v2'  # 22.7M params

model_name = 'cross-encoder/ms-marco-MiniLM-L4-v2'
model_name_short = 'MiniLM-L4-v2'

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CrossEncoder(model_name)

from datasets import load_from_disk
dataset = load_from_disk('coding_dataset')
# train_dataset = dataset['train']
validation_dataset = dataset['validation']

# R-Precision metric

## Evaluation functions

In [3]:
from typing import Generator

results = {}

def mark_as_correct(index, score=1.0):
	results[str(index)] = score

def mark_as_incorrect(index):
	results[str(index)] = False

def get_accuracy():
	total_evaluated = len(results)
	total_correct = sum(results.values())
	print(f'Total evaluated: {total_evaluated}, Total correct: {total_correct}')
	return total_correct / total_evaluated if total_evaluated > 0 else 0

def select_paragraphs(query: str, paragraphs: list[str], top_k: int) -> Generator[int, None, None]:
	for result in model.rank(query, paragraphs, top_k):
		yield result['corpus_id']

def evaluate_selections(selected_indices, correct_indices, val_index):
	total_correct = len(correct_indices)
	correct_selected = len([index for index in selected_indices if index in correct_indices])
	score = correct_selected / total_correct
	if score:
		mark_as_correct(val_index, score)
	else:
		mark_as_incorrect(val_index)

## Evaluation

In [4]:
subset = 'validation'  # 'test' subset in MARCO doesn't mention the correct answers.

for query_index, row in enumerate(dataset[subset]):
	correct_indices = [index for index, x in enumerate(row['passages']['is_selected']) if x > 0]
	if not correct_indices:
		# No correct indices. We need not select anything.
		mark_as_correct(query_index)
		continue

	# Select indices using the model
	selected_indices = select_paragraphs(row['query'], row['passages']['passage_text'], 
										 top_k=len(correct_indices))
	evaluate_selections(selected_indices, correct_indices, query_index)

if True:
	accuracy = get_accuracy()
	print(f'Accuracy: {accuracy:.2%}')
	print(f'Model: {model_name_short}')

	with open('scores.txt', 'a') as f:
		f.write(f'{model_name_short} - accuracy: {accuracy:.2%}\n')

Total evaluated: 310, Total correct: 237.0
Accuracy: 76.45%
Model: MiniLM-L4-v2
